In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report
import joblib
import optuna
import os
from sklearn.utils.class_weight import compute_sample_weight

# Load data
X_train = pd.read_csv('../../dataset_processed/X_train_final.csv')
y_train = pd.read_csv('../../dataset_processed/y_train_final.csv').values.ravel()

# Calculate sample weights to combat imbalance
sample_weights = compute_sample_weight('balanced', y_train)


## Phase 1: Baseline Model

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['f1_macro', 'recall_macro', 'accuracy']

baseline_model = xgb.XGBClassifier(random_state=42, objective='multi:softmax', num_class=3, eval_metric='mlogloss')

cv_results_base = cross_validate(baseline_model, X_train, y_train, cv=cv, scoring=scoring, params={'sample_weight': sample_weights})
baseline_f1 = cv_results_base['test_f1_macro'].mean()

print(f"Mean CV Macro F1: {baseline_f1:.4f}")
print(f"Mean CV Macro Recall: {cv_results_base['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy: {cv_results_base['test_accuracy'].mean():.4f}")

baseline_model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_base = baseline_model.predict(X_train)
report_base = classification_report(y_train, y_pred_base)
print("\nClassification Report on Full Train Data (Baseline):\n", report_base)

os.makedirs('results', exist_ok=True)
with open('results/baseline_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {baseline_f1:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_base['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_base['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Baseline):\n")
    f.write(report_base)


## Phase 2: Hyperparameter Tuning with Optuna

In [ ]:
def objective(trial):
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    n_estimators = trial.suggest_int('n_estimators', 100, 500)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)
    min_child_weight = trial.suggest_int('min_child_weight', 1, 10)
    gamma = trial.suggest_float('gamma', 0.0, 5.0)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True)
    reg_lambda = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True)
    
    model = xgb.XGBClassifier(
        random_state=42, 
        objective='multi:softmax', 
        num_class=3, 
        eval_metric='mlogloss',
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        n_jobs=-1
    )
    
    cv_results = cross_validate(model, X_train, y_train, cv=cv, scoring='f1_macro', params={'sample_weight': sample_weights})
    return cv_results['test_score'].mean()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30)

print(f"Best Params: {study.best_params}")
print(f"Best Macro F1: {study.best_value:.4f}")


## Phase 3: Tuned Eval & Comparison

In [ ]:
tuned_model = xgb.XGBClassifier(random_state=42, objective='multi:softmax', num_class=3, eval_metric='mlogloss', **study.best_params)

cv_results_tuned = cross_validate(tuned_model, X_train, y_train, cv=cv, scoring=scoring, params={'sample_weight': sample_weights})
tuned_f1 = cv_results_tuned['test_f1_macro'].mean()

print(f"Mean CV Macro F1 (Tuned): {tuned_f1:.4f}")
print(f"Mean CV Macro Recall (Tuned): {cv_results_tuned['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy (Tuned): {cv_results_tuned['test_accuracy'].mean():.4f}")

tuned_model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_tuned = tuned_model.predict(X_train)
report_tuned = classification_report(y_train, y_pred_tuned)
print("\nClassification Report on Full Train Data (Tuned):\n", report_tuned)

with open('results/tuned_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {tuned_f1:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_tuned['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_tuned['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Tuned):\n")
    f.write(report_tuned)

print("=== XGBoost Performance Comparison ===")
print(f"Baseline Macro F1: {baseline_f1:.4f}")
print(f"Tuned Macro F1:    {tuned_f1:.4f}")
print(f"Improvement:       {tuned_f1 - baseline_f1:+.4f}")


## Phase 4: Save Model

In [ ]:
if tuned_f1 > baseline_f1:
    print("Tuned model is better. Saving tuned model...")
    final_model = tuned_model
else:
    print("Baseline model is better or equal. Saving baseline model...")
    final_model = baseline_model

joblib.dump(final_model, 'xgboost_best_model.pkl')
print("Model saved successfully to xgboost_best_model.pkl")
